# Completed Experiment Comparison

The single read-only presentation/analysis entry point for completed v0 Raw, v1 Gamma-or-Skip and v2 local-shadow results. V2 is the measured negative local-shadow experiment; adaptive routing v3 is a separate future study.

**Inputs:** Existing committed compact S20 metrics, support, timing, errors and plots under docs/issue45-s20.

**Outputs:** In-memory displays only. No local runs, weights, datasets or study inference dependencies are needed.

**Prerequisites:** Committed evidence and a notebook kernel with the project dependencies. No training, source exports, detector inference, diagnostic replays or upstream notebook execution occurs.


<a id="section-1"></a>

## 1. Configuration

All inputs come from committed compact evidence. This notebook has no execution opt-ins and does not call the historical evidence-acquisition implementation.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import file_hash, read_json, require_paths
from vehicle_detection.five_class import FIVE_CLASSES

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
REPORT = ROOT / "docs/issue45-s20"
EXPERIMENTS = {"v0": "Raw", "v1": "Gamma-or-Skip", "v2": "Local-shadow (exploratory)"}
require_paths(summary=REPORT / "summary.json", evidence=REPORT / "evidence.json")
for name, expected in read_json(REPORT / "evidence.json").items():
    if Path(name).name != name or file_hash(REPORT / name) != expected:
        raise ValueError("Committed comparison evidence changed")
summary = read_json(REPORT / "summary.json")
assert summary["training_subset"] == "S20" and summary["seed"] == 2026
assert summary["role"] == "development_validation" and not summary["final_test_accessed"]
records = summary["records"]
metrics = {version: read_json(REPORT / f"{version}_metrics.json") for version in EXPERIMENTS}
for version, result in metrics.items():
    assert result == records[version]["native"]
    assert result["inputs"]["training_subset"] == "S20"
    assert result["role"] == "development_validation" and not result["final_test_accessed"]

<a id="section-2"></a>

## 2. Completed S20 Development Results

Same 9,781 training IDs, 13,975 development IDs and original five-class GT/negatives; independently initialized from the same pinned checkpoint, seed 2026, 20 epochs, 640, batch 16. Checkpoints were selected on shared v0 Raw development, then evaluated on each matching policy.

Percentages below are native dataset-level AP. CPU mean/p95 measurements use the same 256 consumed exploration images, eight threads, three repeats and rotated order. They include native quality diagnostics and exclude reads/JPEG/inference/verification equally; they are not deployment latency.


In [ ]:
overall = pd.DataFrame(
    [
        {
            "Version": version,
            "Policy": policy,
            "mAP50 (%)": metrics[version]["overall"]["mAP50"] * 100,
            "mAP50:95 (%)": metrics[version]["overall"]["mAP50_95"] * 100,
            "Selected epoch": records[version]["selected_epoch"],
            "CPU mean (ms)": records[version]["runtime"]["mean_ms"],
            "CPU p95 (ms)": records[version]["runtime"]["p95_ms"],
        }
        for version, policy in EXPERIMENTS.items()
    ]
).set_index("Version")
display(overall.round(4))
classes = pd.DataFrame(
    [
        {
            "Version": version,
            "Class": row["Class"],
            "AP50 (%)": row["mAP50"] * 100,
            "AP50:95 (%)": row["mAP50-95"] * 100,
        }
        for version, result in metrics.items()
        for row in result["classes"]
    ]
)
display(
    classes.pivot(index="Class", columns="Version", values=["AP50 (%)", "AP50:95 (%)"])
    .reindex(FIVE_CLASSES)
    .round(4)
)
display(Image(filename=str(REPORT / "comparison.png")))

<a id="section-3"></a>

## 3. Subgroups and Errors

Subgroups use frozen definitions: original brightness <0.12 (dark), >=0.12 (bright), sharpness <50 (pixel-defined low sharpness), and images containing motorcycle/bicycle boxes smaller than 32² original pixels. Every GT in a subgroup image remains included; these are image subsets, not size-specific AP-small. Subgroups overlap except dark/bright.

The low-sharpness subset has no bicycle GT and only one bus/motorcycle source group each. Its macro covers four observed classes and rare-class effects are inconclusive. Dark average AP rises slightly for v2, but dark motorcycle AP falls. Do not infer universal night/blur/small-target benefit from these averages.


In [ ]:
support = read_json(REPORT / "support.json")
subgroups = ("dark", "bright", "blurred", "small_two_wheeler")
display(
    pd.DataFrame(
        [
            {
                "Subgroup": group,
                "Images": support[group]["images"],
                "Represented classes": len(records["v0"]["subgroups"][group]["classes"]),
                **{
                    version: records[version]["subgroups"][group]["mAP50_95"] * 100
                    for version in EXPERIMENTS
                },
            }
            for group in subgroups
        ]
    )
    .set_index("Subgroup")
    .round(4)
)
print("Subgroup values are AP50:95 percentages, not per-image AP.")
display(
    pd.DataFrame({group: support[group]["class_source_groups"] for group in subgroups}).reindex(
        FIVE_CLASSES
    )
)
print("Native counts at confidence .25 / IoU .50; they do not replace ranked/localized AP.")
display(pd.DataFrame(summary["errors"]["all"]).T)
display(Image(filename=str(REPORT / "errors.png")))

<a id="section-4"></a>

## 4. Separate Historical Evidence

- **Historical S10 trained development:** 4,890 training images; [original report](../../docs/development-five-v1/RESULTS.md). These measurements are not S20 rows above.
- **Pretrained local-shadow pilot:** 256 calibration-exploration images and the frozen pretrained detector; [failed pilot report](../../docs/ISSUE45_RESULTS.md). Its gate failures and hash-bound-but-not-separately-committed declaration remain unchanged. It is not a trained S20 result.
- **S20 execution provenance:** [completed lifecycle and original diagnostic acquisition](../../docs/ISSUE45_S20_EXECUTION.md). Historical commands there describe how evidence was acquired; this notebook only reads that evidence.

The pilot and S20 populations differ, so comparing their effect sizes cannot isolate a causal fine-tuning effect.


<a id="section-5"></a>

## 5. Comparison Summary

V2 has slightly higher mAP50 but lower primary mAP50:95 than both controls, mixed subgroup/class effects and substantially higher preprocessing cost. The completed experiment does not support promoting local-shadow.

These are development, single-seed results. Development selected the checkpoints; no independent confirmation, final-test result or released replacement policy is claimed. The final holdout stays sealed. Future adaptive-routing v3 belongs in a separate issue/PR.
